# 07 - Portfolio valuation and risk aggregation

Objectif: charger un inventaire, valoriser ligne par ligne, puis agreger les prix et greeks par produit, sous-jacent, maturite, strike et portefeuille.

## Imports

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.portfolio.book import PortfolioMarketContext, PortfolioValuationEngine
from src.risk.report import build_portfolio_risk_summary, risk_pivot_table

## Build a demo inventory

In [ ]:
inventory = pd.DataFrame(
    [
        {
            "portfolio": "BOOK-A",
            "source_sheet": "options",
            "product_id": "OPT-CALL-1",
            "product_type": "call",
            "underlying": "AAPL",
            "quantity": 10.0,
            "strike_1": 100.0,
            "time_to_maturity_years": 1.0,
        },
        {
            "portfolio": "BOOK-A",
            "source_sheet": "options",
            "product_id": "OPT-SPREAD-1",
            "product_type": "call spread",
            "underlying": "AAPL",
            "quantity": 6.0,
            "strike_1": 95.0,
            "strike_2": 110.0,
            "time_to_maturity_years": 0.75,
        },
        {
            "portfolio": "BOOK-B",
            "source_sheet": "structured_notes",
            "product_id": "SN-CPN-1",
            "product_type": "capital protected note",
            "underlying": "MSFT",
            "quantity": 100.0,
            "participation_rate": 0.9,
            "time_to_maturity_years": 1.5,
        },
        {
            "portfolio": "BOOK-B",
            "source_sheet": "structured_notes",
            "product_id": "SN-RC-1",
            "product_type": "reverse convertible",
            "underlying": "MSFT",
            "quantity": 100.0,
            "coupon_rate": 0.12,
            "time_to_maturity_years": 1.0,
        },
        {
            "portfolio": "BOOK-C",
            "source_sheet": "swaps",
            "product_id": "UNSUPPORTED-SWAP",
            "product_type": "swap",
            "underlying": "EURIBOR",
            "quantity": 1000000.0,
            "time_to_maturity_years": 2.0,
        },
    ]
)

inventory

## Portfolio valuation line by line

In [ ]:
engine = PortfolioValuationEngine()
context = PortfolioMarketContext(
    default_spot=100.0,
    rate=0.03,
    volatility=0.20,
    dividend_yield=0.0,
    spot_by_underlying={"AAPL": 105.0, "MSFT": 115.0},
)

result = engine.value_inventory(inventory, market=context)
line_valuations = result.line_valuations

line_valuations

In [ ]:
line_valuations[[
    "line_index",
    "portfolio",
    "product_id",
    "product_type",
    "underlying",
    "status",
    "price",
    "delta",
    "gamma",
    "vega",
    "rho",
]]

## Aggregated valuation tables

In [ ]:
result.by_product

In [ ]:
result.by_underlying

In [ ]:
result.by_maturity

In [ ]:
result.by_strike

In [ ]:
result.by_portfolio

## Risk summary

In [ ]:
risk_summary = build_portfolio_risk_summary(line_valuations)
risk_summary.by_product

## Pivot tables

In [ ]:
pivot_delta = risk_pivot_table(
    line_valuations,
    index="underlying",
    columns="portfolio",
    value="delta",
)
pivot_vega = risk_pivot_table(
    line_valuations,
    index="underlying",
    columns="portfolio",
    value="vega",
)

pivot_delta

In [ ]:
pivot_vega

## Risk heatmaps

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

im0 = axes[0].imshow(pivot_delta.values, aspect="auto", cmap="RdBu")
axes[0].set_title("Delta heatmap (Underlying x Portfolio)")
axes[0].set_xticks(range(len(pivot_delta.columns)))
axes[0].set_xticklabels(pivot_delta.columns)
axes[0].set_yticks(range(len(pivot_delta.index)))
axes[0].set_yticklabels(pivot_delta.index)
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(pivot_vega.values, aspect="auto", cmap="viridis")
axes[1].set_title("Vega heatmap (Underlying x Portfolio)")
axes[1].set_xticks(range(len(pivot_vega.columns)))
axes[1].set_xticklabels(pivot_vega.columns)
axes[1].set_yticks(range(len(pivot_vega.index)))
axes[1].set_yticklabels(pivot_vega.index)
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

## Contribution by product and underlying

In [ ]:
supported = line_valuations[line_valuations["status"] == "supported"].copy()
total_price = supported["price"].sum()

contrib_product = result.by_product[["product_type", "price"]].copy()
contrib_product["contribution_pct"] = np.where(total_price != 0.0, 100.0 * contrib_product["price"] / total_price, 0.0)
contrib_product.sort_values("contribution_pct", ascending=False)

In [ ]:
contrib_underlying = result.by_underlying[["underlying", "price"]].copy()
contrib_underlying["contribution_pct"] = np.where(total_price != 0.0, 100.0 * contrib_underlying["price"] / total_price, 0.0)
contrib_underlying.sort_values("contribution_pct", ascending=False)

In [ ]:
unsupported = line_valuations[line_valuations["status"] != "supported"]
unsupported[["product_id", "product_type", "error_message"]]